# 🪤 TutorTrap
## *Challenge the tutor. Reveal the misconception.*

**TutorTrap** is an AI-powered diagnostic learning experience for introductory physics.

Unlike generic tutors that mark answers right or wrong, TutorTrap:
1. Presents a **plausible but incorrect** educational claim
2. Analyzes **why** the student thinks the way they do
3. Diagnoses the **underlying misconception**
4. Delivers a **targeted follow-up** to challenge that specific misconception
5. Measures whether the misconception was actually **resolved**

### System Architecture
```
Canonical concept truth
        ↓
AI Claim Generator  →  AI Claim Validator  →  Student
                                                  ↓
                                    AI Reasoning Diagnostician
                                                  ↓
                                    AI Intervention Generator
                                                  ↓
                                         Student (follow-up)
                                                  ↓
                                       AI Recovery Check
                                                  ↓
                                    Adaptive Learner Model Update
```

**Tech stack:** Python · Groq API · Gradio · Adaptive learner model (transparent Python logic)

**Honest description:** Generative AI + explainable adaptive learner modeling — not a black-box ML system.

## Install Dependencies

In [1]:
# Install required packages
# Run this cell once; restart kernel if prompted
import subprocess, sys

packages = ["groq", "gradio"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ Dependencies installed.")

✅ Dependencies installed.


##  Imports and Configuration

In [2]:
import os
import json
import time
import re
import copy
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import gradio as gr
from groq import Groq
from datetime import datetime

# ─── API KEY ────────────────────────────────────────────────────────────────
# Set via environment variable:  export GROQ_API_KEY="gsk_..."
# Or paste directly into the string below (do NOT commit to git):
# GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")   # ← paste key here if needed
import os

os.environ["GROQ_API_KEY"] = "Enter your api key here"
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise EnvironmentError("GROQ_API_KEY is not configured.")

client = Groq(api_key=GROQ_API_KEY)
# if not GROQ_API_KEY:
#     raise EnvironmentError(
#         "GROQ_API_KEY not set. "
#         "Set the environment variable or paste your key into GROQ_API_KEY above."
#     )

# print(" Imports OK.")
# print(f"   API key loaded ({len(GROQ_API_KEY)} chars, starts with {GROQ_API_KEY[:6]}…)")

##  Groq Client + Model Discovery

In [4]:
client = Groq(api_key=GROQ_API_KEY)


# ============================================================
# 🤖 Groq Model Selection
# ============================================================

MODEL_PREFERENCES = ['allam-2-7b', 'canopylabs/orpheus-arabic-saudi', 'canopylabs/orpheus-v1-english', 'groq/compound', 'groq/compound-mini', 'meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'openai/gpt-oss-safeguard-20b', 'qwen/qwen3.6-27b', 'qwen/qwen3.8-27b', 'whisper-large-v3', 'whisper-large-v3-turbo']


def discover_model():
    """
    Select a known generative chat model.

    IMPORTANT:
    Do not fall back to arbitrary 'llama'/'gemma' models because
    Groq also exposes specialized classifier/safety models.
    """

    available = {
        model.id
        for model in client.models.list().data
    }

    for preferred in MODEL_PREFERENCES:
        if preferred in available:
            return preferred

    raise RuntimeError(
        "No supported TutorTrap generation model is available.\n"
        f"Available models:\n{sorted(available)}"
    )


MODEL = discover_model()

print(f"✅ Selected generation model: {MODEL}")



# ─── LLM helper ─────────────────────────────────────────────────────────────
def llm(system: str, user: str, temperature: float = 0.7, max_tokens: int = 1024) -> str:
    """Call the LLM; return the text content. Raises on hard failure."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user",   "content": user}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content.strip()

def llm_json(
    system: str,
    user: str,
    required_keys: list,
    temperature: float = 0.4,
    retries: int = 3
) -> dict:
    """Call LLM, extract JSON, validate required keys, and retry safely."""

    raw = ""
    last_error = None

    for attempt in range(retries):
        try:
            raw = llm(
                system,
                user,
                temperature=temperature,
                max_tokens=1024
            )

            # Remove markdown code fences if present
            cleaned = raw.strip()

            if cleaned.startswith("```"):
                cleaned = re.sub(
                    r"^```(?:json)?\s*",
                    "",
                    cleaned,
                    flags=re.IGNORECASE
                )
                cleaned = re.sub(
                    r"\s*```$",
                    "",
                    cleaned
                ).strip()

            # Extract JSON object from any surrounding text
            start = cleaned.find("{")
            end = cleaned.rfind("}")

            if start == -1 or end == -1 or end <= start:
                raise ValueError("No JSON object found in model response.")

            json_str = cleaned[start:end + 1]

            data = json.loads(json_str)

            missing = [
                key for key in required_keys
                if key not in data
            ]

            if missing:
                raise ValueError(
                    f"Missing required keys: {missing}"
                )

            return data

        except Exception as e:
            last_error = e

            print(
                f"   ⚠ LLM JSON attempt "
                f"{attempt + 1}/{retries} failed: {e}"
            )

            if raw:
                print(f"      Raw response: {raw[:250]}")

            if attempt < retries - 1:
                time.sleep(0.75)

    raise RuntimeError(
        f"LLM JSON call failed after {retries} attempts. "
        f"Last error: {last_error}"
    )

print("✅ LLM helpers ready.")
print("API key exists:", bool(GROQ_API_KEY))
print("Selected model:", MODEL)

✅ Selected generation model: allam-2-7b
✅ LLM helpers ready.
API key exists: True
Selected model: allam-2-7b


In [5]:
test = llm(
    "You are a helpful assistant.",
    "Reply with exactly: TutorTrap online.",
    temperature=0.1,
    max_tokens=20
)

print(test)

TutorTrap online.


In [6]:
test_json = llm_json(
    "Return only valid JSON.",
    'Return {"status": "ok"}',
    ["status"],
    temperature=0.1
)

print(test_json)

{'status': 'ok'}


##  Concept Library

Each concept includes a canonical truth and common misconception seeds.  
The AI claim validator checks generated claims against this canonical truth.

In [7]:

CONCEPTS = {

    # ============================================================
    # 1. GRAVITY & FALLING OBJECTS
    # ============================================================

    "Gravity & Falling Objects": {
        "canonical_truth":
            "In the absence of air resistance, all objects fall "
            "with the same acceleration due to gravity (~9.8 m/s²), "
            "regardless of mass.",

        "common_misconceptions": [
            "heavier objects fall faster",
            "gravity pulls harder on heavy objects causing faster fall",
            "an object with more mass has a greater gravitational acceleration"
        ],

        "fallback_claim":
            "A heavier object falls faster than a lighter object "
            "because gravity pulls harder on it.",

        "fallback_misconception":
            "Greater gravitational force automatically means greater acceleration"
    },


    # ============================================================
    # 2. NEWTON'S FIRST LAW
    # ============================================================

    "Newton's First Law (Inertia)": {
        "canonical_truth":
            "An object remains at rest or moves at constant velocity "
            "unless acted upon by a net external force. Moving objects "
            "do not need a continuous force to keep moving.",

        "common_misconceptions": [
            "moving objects need a force to keep moving",
            "motion implies an active force",
            "an object stops because its force runs out"
        ],

        "fallback_claim":
            "A moving object needs a continuous force acting on it "
            "to keep moving at a constant speed.",

        "fallback_misconception":
            "Continuous motion requires continuous force"
    },


    # ============================================================
    # 3. NEWTON'S SECOND LAW
    # ============================================================

    "Newton's Second Law (F=ma)": {
        "canonical_truth":
            "The net force on an object equals its mass times its "
            "acceleration (F=ma). For the same net force, increasing "
            "mass decreases acceleration.",

        "common_misconceptions": [
            "more force always means more speed",
            "more force means an object must be moving faster",
            "heavier objects always accelerate less",
            "acceleration is the same as velocity"
        ],

        "fallback_claim":
            "If an object experiences twice as much net force, "
            "it will eventually move at twice the speed.",

        "fallback_misconception":
            "Force directly determines speed rather than acceleration"
    },


    # ============================================================
    # 4. MASS VS WEIGHT
    # ============================================================

    "Mass vs Weight": {
        "canonical_truth":
            "Mass is the amount of matter in an object and is measured "
            "in kilograms. Weight is the gravitational force on that "
            "mass and is measured in newtons. Mass remains constant "
            "while weight can change with gravitational field strength.",

        "common_misconceptions": [
            "mass and weight are the same thing",
            "mass changes on the Moon",
            "weight is measured in kilograms",
            "a heavier object necessarily has more mass in every context"
        ],

        "fallback_claim":
            "An astronaut has less mass on the Moon because the Moon "
            "has weaker gravity.",

        "fallback_misconception":
            "Mass changes when gravitational field strength changes"
    },


    # ============================================================
    # 5. ENERGY CONSERVATION
    # ============================================================

    "Energy Conservation": {
        "canonical_truth":
            "Energy cannot be created or destroyed; it is transferred "
            "or transformed between forms. In a closed system, total "
            "energy remains constant.",

        "common_misconceptions": [
            "energy gets used up",
            "machines create energy",
            "stored energy disappears when released",
            "friction destroys energy"
        ],

        "fallback_claim":
            "When a moving object stops because of friction, "
            "the energy it had is destroyed.",

        "fallback_misconception":
            "Energy disappears instead of being transformed or transferred"
    },


    # ============================================================
    # 6. DENSITY & FLOATING
    # ============================================================

    "Density & Floating": {
        "canonical_truth":
            "Whether an object floats depends on its average density "
            "relative to the fluid. An object floats when its average "
            "density is less than the density of the fluid.",

        "common_misconceptions": [
            "heavy objects always sink",
            "large objects always sink",
            "small objects always float",
            "floating depends only on mass"
        ],

        "fallback_claim":
            "A large piece of wood must sink because its greater size "
            "means it is heavier than a small piece of wood.",

        "fallback_misconception":
            "Floating depends primarily on mass or size rather than density"
    },


    # ============================================================
    # 7. MOMENTUM
    # ============================================================

    "Momentum": {
        "canonical_truth":
            "Momentum is p = mv. In a closed system with no net external "
            "impulse, total momentum is conserved. Momentum depends on "
            "both mass and velocity.",

        "common_misconceptions": [
            "faster objects always have more momentum",
            "momentum depends only on speed",
            "heavier objects always have more momentum",
            "stationary objects have no mass so have no momentum"
        ],

        "fallback_claim":
            "A faster object always has more momentum than a slower "
            "object, regardless of their masses.",

        "fallback_misconception":
            "Momentum depends only on velocity"
    },


    # ============================================================
    # 8. FRICTION
    # ============================================================

    "Friction": {
        "canonical_truth":
            "Friction is a force that opposes relative motion or the "
            "tendency of surfaces to move relative to one another. "
            "For simple kinetic friction, its magnitude depends on the "
            "normal force and coefficient of friction.",

        "common_misconceptions": [
            "friction always acts opposite the direction of travel",
            "friction always slows an object to a stop",
            "smoother surfaces always have less friction",
            "friction depends directly on contact area"
        ],

        "fallback_claim":
            "Friction always acts backward on a moving object and "
            "therefore always makes the object slow down.",

        "fallback_misconception":
            "Friction always acts directly opposite motion and necessarily slows the object"
    },


    # ============================================================
    # 9. HEAT VS TEMPERATURE
    # ============================================================

    "Heat vs Temperature": {
        "canonical_truth":
            "Temperature is related to the average kinetic energy of "
            "particles. Heat is energy transferred because of a "
            "temperature difference. Temperature and heat are different "
            "physical concepts.",

        "common_misconceptions": [
            "heat and temperature are the same thing",
            "a hotter object always contains more thermal energy",
            "cold means an object contains no thermal energy",
            "temperature measures total thermal energy"
        ],

        "fallback_claim":
            "A cup of boiling water contains more thermal energy than "
            "a bathtub of warm water simply because its temperature is higher.",

        "fallback_misconception":
            "Temperature directly measures total thermal energy"
    },


    # ============================================================
    # 10. PRESSURE
    # ============================================================

    "Pressure": {
        "canonical_truth":
            "Pressure is force per unit area. For a given force, "
            "concentrating that force over a smaller area increases "
            "pressure. In fluids, pressure acts in all directions.",

        "common_misconceptions": [
            "heavier objects always create more pressure",
            "pressure only acts downward",
            "pressure depends only on force",
            "large contact area creates greater pressure"
        ],

        "fallback_claim":
            "A person standing on two feet produces more pressure "
            "on the floor than standing on one foot because more of "
            "their body is touching the floor.",

        "fallback_misconception":
            "Greater contact area means greater pressure for the same force"
    },


    # ============================================================
    # 11. ELECTRIC CIRCUITS
    # ============================================================

    "Electric Circuits": {
        "canonical_truth":
            "In a closed circuit, electric current is the rate of charge "
            "flow. A potential difference drives current, and the current "
            "depends on circuit properties such as resistance.",

        "common_misconceptions": [
            "current gets used up by a bulb",
            "battery sends current that disappears inside components",
            "current is consumed by resistors",
            "a bulb uses up electrons"
        ],

        "fallback_claim":
            "A light bulb uses up some of the current flowing through it, "
            "so less current remains after the bulb.",

        "fallback_misconception":
            "Current is consumed by circuit components"
    },


    # ============================================================
    # 12. VOLTAGE VS CURRENT
    # ============================================================

    "Voltage vs Current": {
        "canonical_truth":
            "Voltage is an electric potential difference, while current "
            "is the rate of charge flow. They are related but are not the "
            "same physical quantity.",

        "common_misconceptions": [
            "voltage and current are the same thing",
            "higher voltage means higher current in every circuit",
            "current is stored inside a battery",
            "voltage is the amount of charge flowing"
        ],

        "fallback_claim":
            "A battery with twice the voltage always produces exactly "
            "twice the current, regardless of the circuit connected to it.",

        "fallback_misconception":
            "Voltage directly determines current without considering resistance"
    },


    # ============================================================
    # 13. WAVES
    # ============================================================

    "Waves": {
        "canonical_truth":
            "A mechanical wave transfers energy and information through "
            "a medium without requiring the medium as a whole to travel "
            "with the wave. Wave speed depends on the medium and wave "
            "properties.",

        "common_misconceptions": [
            "wave motion means the medium travels with the wave",
            "waves always carry matter from one place to another",
            "larger amplitude means faster wave speed",
            "frequency and speed are always the same thing"
        ],

        "fallback_claim":
            "When a water wave travels across a pool, the water itself "
            "travels all the way across the pool with the wave.",

        "fallback_misconception":
            "A wave transports the medium itself over the entire distance"
    },


    # ============================================================
    # 14. LIGHT & REFLECTION
    # ============================================================

    "Light & Reflection": {
        "canonical_truth":
            "Light reflects from a surface such that the angle of "
            "incidence equals the angle of reflection, measured relative "
            "to the normal.",

        "common_misconceptions": [
            "the reflection angle is measured from the surface",
            "rough surfaces do not reflect light",
            "light reflects only from mirrors",
            "a reflected ray always travels directly backward"
        ],

        "fallback_claim":
            "If a light ray hits a mirror at an angle of 30° to the "
            "surface, it reflects at an angle of 30° to the normal.",

        "fallback_misconception":
            "Reflection angles are measured inconsistently from the surface rather than the normal"
    },


    # ============================================================
    # 15. SIMPLE MACHINES & WORK
    # ============================================================

    "Work & Simple Machines": {
        "canonical_truth":
            "Mechanical work is done when a force causes displacement "
            "in the direction of the force. Simple machines can trade "
            "force for distance, but in an ideal system they do not "
            "create energy.",

        "common_misconceptions": [
            "using a machine creates energy",
            "larger force always means more work",
            "no movement still means work was done",
            "a machine reduces the total energy required"
        ],

        "fallback_claim":
            "A ramp allows you to lift an object with less effort, "
            "so less total work is needed to raise it to the same height.",

        "fallback_misconception":
            "Reducing the required force also reduces the total work in an ideal machine"
    },
}

CONCEPT_NAMES = list(CONCEPTS.keys())
print(f"✅ Concept library loaded: {len(CONCEPTS)} concepts")
for name in CONCEPT_NAMES:
    print(f"   • {name}")

✅ Concept library loaded: 2 concepts
   • Gravity & Falling Objects
   • Newton's First Law (Inertia)


##  Adaptive Learner Model

A transparent, explainable Python learner model.  
This is **not** described as machine learning — it uses simple, inspectable logic.

In [8]:
def init_learner() -> dict:
    """Initialize a fresh learner state."""
    return {
        "mastery": {c: 0.5 for c in CONCEPT_NAMES},   # 0.0–1.0 per concept
        "misconceptions": {},   # concept → list of detected misconception strings
        "misconception_confidence": {},  # concept → float 0–1
        "resolved": {},         # concept → bool
        "attempts": [],         # list of attempt dicts
        "session_start": datetime.now().isoformat(),
    }


def update_learner(learner: dict, concept: str, diagnosis: dict, resolved: bool) -> dict:
    """
    Update learner state after one complete TutorTrap interaction.

    Rules (transparent logic):
    - If misconception found and student was wrong: decrease mastery, record misconception.
    - If resolved: increase mastery, mark resolved.
    - Mastery is bounded [0.0, 1.0].
    """
    learner = copy.deepcopy(learner)
    current_mastery = learner["mastery"].get(concept, 0.5)
    miscon_label = diagnosis.get("misconception", "")
    student_correct = diagnosis.get("student_correct", False)
    confidence = float(diagnosis.get("misconception_confidence", 0.5))

    # Record misconception
    if miscon_label and not student_correct:
        if concept not in learner["misconceptions"]:
            learner["misconceptions"][concept] = []
        if miscon_label not in learner["misconceptions"][concept]:
            learner["misconceptions"][concept].append(miscon_label)
        learner["misconception_confidence"][concept] = confidence

    # Update mastery
    if resolved:
        delta = +0.25
        learner["resolved"][concept] = True
        learner["misconception_confidence"][concept] = max(
            0.0, learner["misconception_confidence"].get(concept, confidence) - 0.6
        )
    elif student_correct and not miscon_label:
        delta = +0.15
    elif student_correct:
        delta = +0.05
    else:
        delta = -0.10

    new_mastery = max(0.0, min(1.0, current_mastery + delta))
    learner["mastery"][concept] = new_mastery

    # Record attempt
    learner["attempts"].append({
        "concept": concept,
        "student_correct": student_correct,
        "misconception": miscon_label,
        "resolved": resolved,
        "mastery_before": current_mastery,
        "mastery_after": new_mastery,
        "timestamp": datetime.now().isoformat(),
    })

    return learner


def select_next_concept(learner: dict) -> str:
    """
    Adaptive selection: prioritize concepts with:
    1. Unresolved misconceptions (highest priority)
    2. Lowest mastery score
    """
    # Find concepts with unresolved misconceptions
    unresolved = [
        c for c in CONCEPT_NAMES
        if c in learner["misconceptions"]
        and not learner["resolved"].get(c, False)
    ]
    if unresolved:
        # Pick the one with highest misconception confidence
        return max(unresolved, key=lambda c: learner["misconception_confidence"].get(c, 0))

    # Otherwise pick the concept with lowest mastery
    return min(CONCEPT_NAMES, key=lambda c: learner["mastery"].get(c, 0.5))


# Initialize global learner state
LEARNER = init_learner()
print("✅ Adaptive learner model initialized.")
print(f"   Starting mastery (all concepts): {next(iter(LEARNER['mastery'].values())):.0%} (uniform prior)")

✅ Adaptive learner model initialized.
   Starting mastery (all concepts): 50% (uniform prior)


## AI Component 1: Claim Generator

In [9]:
CLAIM_GEN_SYSTEM = """
You are TutorTrap's claim generator for introductory physics education.
Your job is to generate a SINGLE plausible but INCORRECT educational claim 
that embeds a common student misconception about the given physics concept.

Requirements:
- The claim must sound like something a plausible (but wrong) teacher might say
- It must be factually INCORRECT
- It must be related to a REAL, COMMON misconception students have
- It must be concise: one or two sentences maximum
- Do NOT add disclaimers or explanations — just the claim itself
- Do NOT say 'incorrect' or hint that it's wrong

Respond with ONLY a JSON object:
{"claim": "<the plausible incorrect claim>", "target_misconception": "<brief name of the misconception>"}
"""

def generate_claim(concept: str) -> dict:
    """Generate a misconception-based claim for a given concept."""
    canonical = CONCEPTS[concept]["canonical_truth"]
    common = CONCEPTS[concept]["common_misconceptions"]

    user_prompt = f"""
Concept: {concept}
Canonical truth (the CORRECT answer): {canonical}
Common misconception seeds to draw from: {', '.join(common)}

Generate a plausible but INCORRECT claim that a student with this misconception might believe.
Return only valid JSON.
"""
    return llm_json(
        system=CLAIM_GEN_SYSTEM,
        user=user_prompt,
        required_keys=["claim", "target_misconception"],
        temperature=0.8,
    )

print("✅ AI Claim Generator ready.")

✅ AI Claim Generator ready.


##  AI Component 2: Claim Validator

Before showing any claim to a student, the validator checks it against the canonical truth.  
This ensures TutorTrap never accidentally teaches an incorrect claim as correct.

In [10]:
CLAIM_VALIDATOR_SYSTEM = """
You are a physics education quality checker.
You will receive a canonical scientific truth, and a generated claim.
Your job is to verify the claim meets ALL of these criteria:

1. scientifically_incorrect: The claim contradicts the canonical truth
2. plausible: A real student might genuinely believe this
3. on_topic: The claim is about the stated concept
4. not_nonsensical: The claim makes grammatical and logical sense

Respond with ONLY valid JSON:
{"valid": true/false, "reasons": ["reason1", ...], "verdict": "<one sentence verdict>"}
"""

def validate_claim(concept: str, claim: str) -> dict:
    """Validate a generated claim before showing it to the student."""
    canonical = CONCEPTS[concept]["canonical_truth"]
    user_prompt = f"""
Concept: {concept}
Canonical truth: {canonical}
Generated claim to validate: "{claim}"

Is this claim suitable for TutorTrap? Return only valid JSON.
"""
    return llm_json(
        system=CLAIM_VALIDATOR_SYSTEM,
        user=user_prompt,
        required_keys=["valid", "verdict"],
        temperature=0.2,
    )


def generate_validated_claim(
    concept: str,
    max_attempts: int = 3
) -> dict:

    for attempt in range(max_attempts):

        try:
            claim_data = generate_claim(concept)

            validation = validate_claim(
                concept,
                claim_data["claim"]
            )

            if validation.get("valid", False):

                claim_data["validation"] = validation

                return claim_data

            print(
                f"   ⚠ Claim rejected "
                f"(attempt {attempt + 1}): "
                f"{validation.get('verdict', '')}"
            )

        except Exception as e:

            print(
                f"   ⚠ Claim generation failure "
                f"(attempt {attempt + 1}): {e}"
            )

    fallback = {
        "claim": CONCEPTS[concept]["fallback_claim"],
        "target_misconception":
            CONCEPTS[concept]["fallback_misconception"],
        "fallback": True,
    }

    print(
        f"   ⚠ Using concept-specific fallback "
        f"for '{concept}'."
    )

    return fallback

print("✅ AI Claim Validator ready.")

✅ AI Claim Validator ready.


##  AI Component 3: Reasoning Diagnostician

In [11]:
DIAGNOSTICIAN_SYSTEM = """
You are TutorTrap's reasoning diagnostician — an expert physics educator.
You analyze a student's response to a physics claim to:
  1. Determine whether they identified the claim as incorrect
  2. Identify the specific misconception in their reasoning
  3. Find evidence of the misconception in their own words
  4. Provide supportive, non-condescending feedback

IMPORTANT:
- student_correct = true means they correctly identified the claim as FALSE (or correctly disagreed)
- student_correct = false means they agreed with a false claim OR disagreed with wrong reasoning
- misconception_confidence should be 0.0 if student was fully correct, up to 1.0 if strongly wrong
- estimated_mastery: your estimate of this student's concept mastery (0.0 to 1.0)

Respond with ONLY valid JSON:
{
  "student_correct": true/false,
  "misconception": "<brief name or empty string if none>",
  "misconception_confidence": 0.0–1.0,
  "evidence_from_reasoning": "<quote or paraphrase from student showing the misconception>",
  "diagnosis": "<2-3 sentence explanation of what the student misunderstands>",
  "supportive_feedback": "<encouraging 1-sentence acknowledgment of what they got right>",
  "estimated_mastery": 0.0–1.0
}
"""

def diagnose_reasoning(concept: str, claim: str, student_agreed: bool, explanation: str) -> dict:
    """Analyze student reasoning and identify the underlying misconception."""
    canonical = CONCEPTS[concept]["canonical_truth"]
    agreement_str = "agreed with the claim" if student_agreed else "disagreed with the claim"

    user_prompt = f"""
Concept: {concept}
Canonical truth: {canonical}
Claim shown to student (INCORRECT): "{claim}"
Student response: They {agreement_str}.
Student's written explanation: "{explanation}"

Analyze this student's reasoning. Return only valid JSON.
"""
    return llm_json(
        system=DIAGNOSTICIAN_SYSTEM,
        user=user_prompt,
        required_keys=["student_correct", "misconception", "diagnosis"],
        temperature=0.3,
    )


print("✅ AI Reasoning Diagnostician ready.")

✅ AI Reasoning Diagnostician ready.


##  AI Component 4: Intervention Generator

In [12]:
INTERVENTION_SYSTEM = """
You are TutorTrap's intervention designer.
Given a specific misconception a student has, generate ONE targeted follow-up question.

The question must:
- Directly challenge the identified misconception
- Be concrete and scenario-based (not abstract)
- Be answerable in 2-4 sentences by a student
- Guide the student toward the correct understanding without giving it away
- Be engaging and feel relevant

Respond with ONLY valid JSON:
{"intervention_question": "<the targeted question>", "what_this_tests": "<brief: what understanding this checks for>"}
"""

def generate_intervention(concept: str, misconception: str, diagnosis: str) -> dict:
    """Generate a targeted follow-up question for the diagnosed misconception."""
    user_prompt = f"""
Concept: {concept}
Identified misconception: {misconception}
Diagnosis: {diagnosis}

Generate a targeted follow-up question that will test whether the student can overcome this misconception.
Return only valid JSON.
"""
    return llm_json(
        system=INTERVENTION_SYSTEM,
        user=user_prompt,
        required_keys=["intervention_question"],
        temperature=0.6,
    )


print("✅ AI Intervention Generator ready.")

✅ AI Intervention Generator ready.


##  AI Component 5: Recovery Check

This is the key component that shows **misconception → intervention → resolution**.  
It determines whether the targeted intervention actually changed the student's understanding.

In [13]:
RECOVERY_SYSTEM = """
You are TutorTrap's misconception resolution checker.
A student previously showed a misconception. They were then given a targeted follow-up question.
Your job is to evaluate whether their new answer indicates the misconception has been resolved.

Be honest and rigorous:
- resolved=true: The student clearly demonstrates correct understanding
- resolved=false: The student still shows the same (or a related) misconception
- resolved=partial: The student shows improvement but not full understanding

Respond with ONLY valid JSON:
{
  "resolved": true/false,
  "resolution_status": "resolved" | "partial" | "unresolved",
  "explanation": "<2-3 sentences on what their answer reveals>",
  "updated_misconception_confidence": 0.0–1.0,
  "estimated_mastery_change": "increased" | "unchanged" | "decreased"
}
"""

def check_recovery(
    concept: str,
    original_misconception: str,
    intervention_question: str,
    student_followup_answer: str,
) -> dict:
    """Check whether the misconception appears resolved after intervention."""
    canonical = CONCEPTS[concept]["canonical_truth"]
    user_prompt = f"""
Concept: {concept}
Canonical truth: {canonical}
Original misconception: {original_misconception}
Targeted intervention question asked: "{intervention_question}"
Student's follow-up answer: "{student_followup_answer}"

Has the misconception been resolved? Return only valid JSON.
"""
    return llm_json(
        system=RECOVERY_SYSTEM,
        user=user_prompt,
        required_keys=["resolved", "resolution_status", "explanation", "updated_misconception_confidence"],
        temperature=0.3,
    )


print("✅ AI Recovery Check ready.")

✅ AI Recovery Check ready.


##  Session State Controller

Manages the multi-step TutorTrap interaction as a clean state machine.

In [14]:
def fresh_session_state():
    """Create a fresh session state dictionary."""
    return {
        "concept": None,
        "claim": None,
        "target_misconception": None,
        "student_agreed": None,
        "explanation": None,
        "diagnosis": None,
        "intervention": None,
        "recovery": None,
        "step": "claim",  # claim → reasoning → followup → result
        "mastery_before": None,
        "misconception_confidence_before": None,
    }


# Global mutable session state (used by Gradio callbacks)
SESSION = fresh_session_state()


def start_new_challenge(learner_state):
    """Pick next concept, generate claim, reset SESSION. Returns (claim_text, concept_name, status)."""
    global SESSION
    SESSION = fresh_session_state()

    concept = select_next_concept(learner_state)
    SESSION["concept"] = concept
    SESSION["mastery_before"] = learner_state["mastery"].get(concept, 0.5)
    SESSION["misconception_confidence_before"] = learner_state["misconception_confidence"].get(concept, None)

    claim_data = generate_validated_claim(concept)
    SESSION["claim"] = claim_data["claim"]
    SESSION["target_misconception"] = claim_data["target_misconception"]
    SESSION["step"] = "reasoning"

    return SESSION["claim"], concept


print("✅ Session controller ready.")

✅ Session controller ready.


##  Progress Visualization

In [15]:
def make_progress_chart(learner_state: dict) -> str:
    """
    Generate a horizontal bar chart of concept mastery.
    Saves to a temp file and returns the path (for Gradio Image component).
    """
    import os, tempfile

    concepts = CONCEPT_NAMES
    mastery = [learner_state["mastery"].get(c, 0.5) for c in concepts]
    resolved = [learner_state["resolved"].get(c, False) for c in concepts]
    misconceptions = [bool(learner_state["misconceptions"].get(c)) for c in concepts]

    colors = []
    for c, r, m in zip(concepts, resolved, misconceptions):
        if r:
            colors.append("#22c55e")   # green = resolved
        elif m:
            colors.append("#f97316")   # orange = unresolved misconception
        else:
            colors.append("#60a5fa")   # blue = neutral

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.barh(concepts, mastery, color=colors, edgecolor="white", height=0.6)

    ax.set_xlim(0, 1)
    ax.set_xlabel("Mastery", fontsize=11)
    ax.set_title("📊 Learner Progress — TutorTrap", fontsize=13, fontweight="bold", pad=12)
    ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="y", labelsize=9)

    for bar, val in zip(bars, mastery):
        ax.text(min(val + 0.02, 0.98), bar.get_y() + bar.get_height() / 2,
                f"{val:.0%}", va="center", ha="left", fontsize=9, color="#1e293b")

    legend_patches = [
        mpatches.Patch(color="#22c55e", label="✓ Misconception resolved"),
        mpatches.Patch(color="#f97316", label="⚠ Active misconception"),
        mpatches.Patch(color="#60a5fa", label="◦ No misconception detected"),
    ]
    ax.legend(handles=legend_patches, loc="lower right", fontsize=8, framealpha=0.7)

    fig.tight_layout()
    tmpfile = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    fig.savefig(tmpfile.name, dpi=130, bbox_inches="tight")
    plt.close(fig)
    return tmpfile.name


def format_learner_summary(learner_state: dict) -> str:
    """Return a text summary of learner state for display."""
    lines = []
    total_attempts = len(learner_state["attempts"])
    resolved_count = sum(1 for v in learner_state["resolved"].values() if v)
    misconception_count = len(learner_state["misconceptions"])

    lines.append(f"**Total attempts:** {total_attempts}")
    lines.append(f"**Misconceptions detected:** {misconception_count}")
    lines.append(f"**Misconceptions resolved:** {resolved_count}")
    lines.append("")

    if learner_state["misconceptions"]:
        lines.append("**Detected misconceptions:**")
        for concept, mlist in learner_state["misconceptions"].items():
            status = "✓ resolved" if learner_state["resolved"].get(concept) else "⚠ active"
            conf = learner_state["misconception_confidence"].get(concept, 0)
            lines.append(f"  • **{concept}** [{status}, confidence: {conf:.0%}]")
            for m in mlist:
                lines.append(f"    — {m}")

    return "\n".join(lines)


print("✅ Progress visualization ready.")

✅ Progress visualization ready.


##  Gradio UI

The main TutorTrap interface. Designed as a mini-product, not a chatbot.  
**Run this cell to launch TutorTrap.**

In [16]:


# ─── Custom CSS ─────────────────────────────────────────────────────────────

CUSTOM_CSS = """
.gradio-container {
    max-width: 860px !important;
    margin: auto;
    font-family: 'Segoe UI', system-ui, sans-serif;
}

#claim-box {
    background: #1e293b;
    border-radius: 12px;
    padding: 20px 24px;
    border: 1px solid #334155;
}

#claim-box textarea {
    font-size: 17px !important;
    color: #f1f5f9 !important;
    background: transparent !important;
    border: none !important;
    font-style: italic;
    line-height: 1.6;
}

#miscon-box {
    background: #431407;
    border-radius: 10px;
    padding: 16px;
    border: 1px solid #c2410c;
}

#correct-box {
    background: #14532d;
    border-radius: 10px;
    padding: 16px;
    border: 1px solid #16a34a;
}

#result-box {
    border-radius: 10px;
    padding: 16px;
}

.agree-btn {
    background: #1d4ed8 !important;
    color: white !important;
}

.disagree-btn {
    background: #7c3aed !important;
    color: white !important;
}

h1 {
    font-size: 2rem !important;
    font-weight: 800 !important;
}
"""


# ─── Build UI ────────────────────────────────────────────────────────────────

with gr.Blocks(title="TutorTrap") as app:

    # ------------------------------------------------------------------
    # TITLE
    # ------------------------------------------------------------------

    gr.Markdown("""
# 🪤 TutorTrap
### *Challenge the tutor. Reveal the misconception.*

An AI tutor will make a **plausible but incorrect** physics claim.

Your job: evaluate it, explain your reasoning, and let TutorTrap diagnose your thinking.
""")

    # ------------------------------------------------------------------
    # SESSION STATE
    # ------------------------------------------------------------------

    learner_gr = gr.State(init_learner())
    session_gr = gr.State(fresh_session_state())


    # ------------------------------------------------------------------
    # STEP 1 — AI CLAIM
    # ------------------------------------------------------------------

    with gr.Group():

        gr.Markdown("## 🤖 AI Tutor Claims:")

        concept_display = gr.Markdown(
            "*Click 'New Challenge' to begin*"
        )

        claim_display = gr.Textbox(
            label="The AI Tutor says:",
            interactive=False,
            lines=3,
            placeholder="Click 'New Challenge' to generate a claim…",
            elem_id="claim-box",
        )

        new_challenge_btn = gr.Button(
            "⚡ New Challenge",
            variant="primary",
            size="lg"
        )


    # ------------------------------------------------------------------
    # STEP 2 — STUDENT RESPONSE
    # ------------------------------------------------------------------

    with gr.Group(visible=False) as step2_group:

        gr.Markdown("## 💭 Your Turn")

        with gr.Row():

            agree_btn = gr.Button(
                "👍 Agree",
                variant="secondary",
                size="lg",
                elem_classes=["agree-btn"]
            )

            disagree_btn = gr.Button(
                "👎 Disagree",
                variant="secondary",
                size="lg",
                elem_classes=["disagree-btn"]
            )

        agreement_display = gr.Markdown(
            "*Select Agree or Disagree above*"
        )

        explanation_box = gr.Textbox(
            label="Explain your reasoning:",
            placeholder="Why do you agree or disagree? Explain in 2–4 sentences…",
            lines=4,
        )

        analyze_btn = gr.Button(
            "🔍 Analyze My Reasoning",
            variant="primary",
            size="lg"
        )


    # ------------------------------------------------------------------
    # STEP 3 — DIAGNOSIS
    # ------------------------------------------------------------------

    with gr.Group(visible=False) as step3_group:

        gr.Markdown("## 🧠 TutorTrap Diagnosis")

        diagnosis_display = gr.Markdown("")


    # ------------------------------------------------------------------
    # STEP 4 — INTERVENTION
    # ------------------------------------------------------------------

    with gr.Group(visible=False) as step4_group:

        gr.Markdown("## 🎯 Targeted Challenge")

        intervention_display = gr.Markdown("")

        followup_box = gr.Textbox(
            label="Your answer:",
            placeholder="Answer the challenge question above…",
            lines=4,
        )

        submit_followup_btn = gr.Button(
            "✅ Submit Answer",
            variant="primary",
            size="lg"
        )


    # ------------------------------------------------------------------
    # STEP 5 — RESULT
    # ------------------------------------------------------------------

    with gr.Group(visible=False) as step5_group:

        gr.Markdown("## 📈 Results")

        result_display = gr.Markdown("")

        learner_summary = gr.Markdown("")

        next_btn = gr.Button(
            "⚡ Next Challenge",
            variant="primary",
            size="lg"
        )


    # ------------------------------------------------------------------
    # TEACHER / ANALYTICS PANEL
    # ------------------------------------------------------------------

    with gr.Accordion(
        "📊 Learner Progress (Teacher View)",
        open=False
    ):

        # Gradio 6 compatibility:
        # show_download_button is no longer passed here.
        progress_chart = gr.Image(
            label="Mastery by Concept"
        )

        refresh_chart_btn = gr.Button(
            "🔄 Refresh Chart"
        )


    # ==================================================================
    # CALLBACKS
    # ==================================================================

    # ------------------------------------------------------------------
    # NEW CHALLENGE
    # ------------------------------------------------------------------

    def cb_new_challenge(learner_state, session_state):

        try:

            claim_text, concept = start_new_challenge(
                learner_state
            )

            new_session = copy.deepcopy(SESSION)

            return (
                f"**Concept:** {concept}",
                claim_text,

                gr.update(visible=True),
                gr.update(visible=False),
                gr.update(visible=False),
                gr.update(visible=False),

                "*Select Agree or Disagree above*",

                "",
                "",

                new_session
            )

        except Exception as e:

            return (
                "**Error generating challenge**",

                f"⚠ Error: {e}\n\n"
                "Please check your Groq configuration and try again.",

                gr.update(visible=False),
                gr.update(visible=False),
                gr.update(visible=False),
                gr.update(visible=False),

                "",
                "",
                "",

                session_state
            )


    new_challenge_btn.click(
        fn=cb_new_challenge,

        inputs=[
            learner_gr,
            session_gr
        ],

        outputs=[
            concept_display,
            claim_display,
            step2_group,
            step3_group,
            step4_group,
            step5_group,
            agreement_display,
            explanation_box,
            followup_box,
            session_gr
        ]
    )


    # ------------------------------------------------------------------
    # AGREE
    # ------------------------------------------------------------------

    def cb_agree(session_state):

        session_state = copy.deepcopy(session_state)

        session_state["student_agreed"] = True

        return (
            "✅ **You agreed** with the claim. "
            "Now explain why below.",
            session_state
        )


    agree_btn.click(
        fn=cb_agree,

        inputs=[session_gr],

        outputs=[
            agreement_display,
            session_gr
        ]
    )


    # ------------------------------------------------------------------
    # DISAGREE
    # ------------------------------------------------------------------

    def cb_disagree(session_state):

        session_state = copy.deepcopy(session_state)

        session_state["student_agreed"] = False

        return (
            "❌ **You disagreed** with the claim. "
            "Now explain why below.",
            session_state
        )


    disagree_btn.click(
        fn=cb_disagree,

        inputs=[session_gr],

        outputs=[
            agreement_display,
            session_gr
        ]
    )


    # ------------------------------------------------------------------
    # ANALYZE REASONING
    # ------------------------------------------------------------------

    def cb_analyze(
        explanation,
        learner_state,
        session_state
    ):

        if not explanation or not explanation.strip():

            return (
                gr.update(visible=False),
                "⚠ Please explain your reasoning before analyzing.",
                gr.update(visible=False),
                "",
                learner_state,
                session_state
            )


        if session_state.get("student_agreed") is None:

            return (
                gr.update(visible=False),
                "⚠ Please select Agree or Disagree first.",
                gr.update(visible=False),
                "",
                learner_state,
                session_state
            )


        try:

            session_state = copy.deepcopy(session_state)

            concept = session_state["concept"]
            claim = session_state["claim"]
            agreed = session_state["student_agreed"]

            session_state["explanation"] = explanation


            # ----------------------------------------------------------
            # AI DIAGNOSIS
            # ----------------------------------------------------------

            diagnosis = diagnose_reasoning(
                concept,
                claim,
                agreed,
                explanation
            )

            session_state["diagnosis"] = diagnosis


            # ----------------------------------------------------------
            # DISPLAY DIAGNOSIS
            # ----------------------------------------------------------

            correct = diagnosis.get(
                "student_correct",
                False
            )

            misconception = diagnosis.get(
                "misconception",
                ""
            )

            evidence = diagnosis.get(
                "evidence_from_reasoning",
                ""
            )

            diag_text = diagnosis.get(
                "diagnosis",
                ""
            )

            feedback = diagnosis.get(
                "supportive_feedback",
                ""
            )

            conf = float(
                diagnosis.get(
                    "misconception_confidence",
                    0.0
                )
            )


            if correct and not misconception:

                diag_md = f"""
### ✅ Correct identification!

{feedback}

**What this shows:**
{diag_text}
"""

            else:

                diag_md = f"""
### ⚠ Misconception Detected

**Likely misconception** *(confidence: {conf:.0%})*

> {misconception}

**{feedback}**

**Evidence from your reasoning:**

> *"{evidence}"*

**What this reveals:**

{diag_text}
"""


            # ----------------------------------------------------------
            # TARGETED INTERVENTION
            # ----------------------------------------------------------

            miscon_for_intervention = (
                misconception
                if misconception
                else diagnosis.get(
                    "diagnosis",
                    ""
                )
            )

            intervention = generate_intervention(
                concept,
                miscon_for_intervention,
                diag_text
            )

            session_state["intervention"] = intervention

            intervention_question = intervention[
                "intervention_question"
            ]

            tests_for = intervention.get(
                "what_this_tests",
                ""
            )


            intervention_md = f"""
**TutorTrap Challenge:**

> {intervention_question}

*This tests: {tests_for}*
"""


            return (
                gr.update(visible=True),
                diag_md,
                gr.update(visible=True),
                intervention_md,
                learner_state,
                session_state
            )


        except Exception as e:

            return (
                gr.update(visible=True),
                f"⚠ Analysis error: {e}",
                gr.update(visible=False),
                "",
                learner_state,
                session_state
            )


    analyze_btn.click(
        fn=cb_analyze,

        inputs=[
            explanation_box,
            learner_gr,
            session_gr
        ],

        outputs=[
            step3_group,
            diagnosis_display,
            step4_group,
            intervention_display,
            learner_gr,
            session_gr
        ]
    )


    # ------------------------------------------------------------------
    # FOLLOW-UP / RECOVERY
    # ------------------------------------------------------------------

    def cb_followup(
        followup_answer,
        learner_state,
        session_state
    ):

        if not followup_answer or not followup_answer.strip():

            return (
                gr.update(visible=False),
                "⚠ Please answer the targeted challenge.",
                "",
                learner_state,
                session_state
            )


        try:

            session_state = copy.deepcopy(session_state)

            concept = session_state["concept"]

            diagnosis = session_state["diagnosis"]

            misconception = diagnosis.get(
                "misconception",
                ""
            )

            intervention_q = session_state[
                "intervention"
            ]["intervention_question"]


            mastery_before = session_state[
                "mastery_before"
            ]

            conf_before = session_state[
                "misconception_confidence_before"
            ]

            conf_from_diag = float(
                diagnosis.get(
                    "misconception_confidence",
                    0.5
                )
            )

            display_conf_before = (
                conf_before
                if conf_before is not None
                else conf_from_diag
            )


            # ----------------------------------------------------------
            # AI RECOVERY CHECK
            # ----------------------------------------------------------

            recovery = check_recovery(
                concept,
                misconception,
                intervention_q,
                followup_answer
            )

            session_state["recovery"] = recovery


            resolved = bool(
                recovery.get(
                    "resolved",
                    False
                )
            )

            status = recovery.get(
                "resolution_status",
                "unresolved"
            )

            recovery_explanation = recovery.get(
                "explanation",
                ""
            )

            conf_after = float(
                recovery.get(
                    "updated_misconception_confidence",
                    conf_from_diag
                )
            )


            # ----------------------------------------------------------
            # UPDATE LEARNER
            # ----------------------------------------------------------

            updated_learner = update_learner(
                learner_state,
                concept,
                diagnosis,
                resolved
            )

            mastery_after = updated_learner[
                "mastery"
            ][concept]


            # ----------------------------------------------------------
            # RESULT DISPLAY
            # ----------------------------------------------------------

            if resolved or status == "resolved":

                status_icon = "✅"
                status_label = (
                    "Misconception appears **resolved**!"
                )

            elif status == "partial":

                status_icon = "🔄"
                status_label = (
                    "**Partial improvement** — "
                    "keep exploring!"
                )

            else:

                status_icon = "⚠"
                status_label = (
                    "Misconception **persists** — "
                    "more practice needed."
                )


            result_md = f"""
## {status_icon} Concept Check

**{status_label}**

{recovery_explanation}

---

| Metric | Before | After |
|---|---|---|
| Concept mastery | {mastery_before:.0%} | {mastery_after:.0%} |
| Misconception confidence | {display_conf_before:.0%} | {conf_after:.0%} |

**Concept:** {concept}

**Misconception targeted:** {misconception or '(none detected)'}

**Status:** {status.capitalize()}
"""


            learner_md = format_learner_summary(
                updated_learner
            )


            # Prepare next session

            next_session = copy.deepcopy(
                session_state
            )

            next_session["step"] = "result"


            return (
                gr.update(visible=True),
                result_md,
                learner_md,
                updated_learner,
                next_session
            )


        except Exception as e:

            return (
                gr.update(visible=True),
                f"⚠ Recovery check error: {e}",
                "",
                learner_state,
                session_state
            )


    submit_followup_btn.click(
        fn=cb_followup,

        inputs=[
            followup_box,
            learner_gr,
            session_gr
        ],

        outputs=[
            step5_group,
            result_display,
            learner_summary,
            learner_gr,
            session_gr
        ]
    )


    # ------------------------------------------------------------------
    # NEXT CHALLENGE
    # ------------------------------------------------------------------

    next_btn.click(
        fn=cb_new_challenge,

        inputs=[
            learner_gr,
            session_gr
        ],

        outputs=[
            concept_display,
            claim_display,
            step2_group,
            step3_group,
            step4_group,
            step5_group,
            agreement_display,
            explanation_box,
            followup_box,
            session_gr
        ]
    )


    # ------------------------------------------------------------------
    # REFRESH CHART
    # ------------------------------------------------------------------

    def cb_refresh_chart(learner_state):
        return make_progress_chart(
            learner_state
        )


    refresh_chart_btn.click(
        fn=cb_refresh_chart,

        inputs=[learner_gr],

        outputs=[progress_chart]
    )


# ======================================================================
# LAUNCH
# ======================================================================

print("🚀 Launching TutorTrap UI…")

app.launch(
    share=False,
    inbrowser=True,
    css=CUSTOM_CSS,
    theme=gr.themes.Soft()
)

🚀 Launching TutorTrap UI…
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


   ⚠ Claim rejected (attempt 1): The claim is not suitable because it mentions air resistance and does not follow the concept of acceleration due to gravity solely based on mass.
   ⚠ Claim rejected (attempt 2): The claim does not consider air resistance and states that acceleration depends on mass, not the force of gravity.
   ⚠ Claim rejected (attempt 3): The claim does not accurately represent the concept of gravity and falling objects, as heavier objects do not fall faster due to their mass but rather experience stronger air resistance
   ⚠ Using concept-specific fallback for 'Gravity & Falling Objects'.


C:\Users\akhil\AppData\Local\Temp\ipykernel_14820\219992370.py:44: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from current font.
  fig.tight_layout()
C:\Users\akhil\AppData\Local\Temp\ipykernel_14820\219992370.py:46: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from current font.
  fig.savefig(tmpfile.name, dpi=130, bbox_inches="tight")


## End-to-End Smoke Test

Run this cell **without the UI** to verify all AI components work correctly.  
Useful for debugging before the demo.

In [17]:
def run_smoke_test(concept: str = "Gravity & Falling Objects"):
    print(f"\n{'='*60}")
    print(f"🔬 TutorTrap Smoke Test — {concept}")
    print(f"{'='*60}\n")

    # 1. Generate & validate claim
    print("[1] Generating validated claim…")
    claim_data = generate_validated_claim(concept)
    claim = claim_data["claim"]
    print(f"    ✅ Claim: {claim}")
    print(f"       Target misconception: {claim_data['target_misconception']}")

    # 2. Simulate student response (agrees with false claim = misconception)
    simulated_explanation = (
        "I agree because heavier objects have more mass, "
        "and since gravity is pulling on more mass, it must pull with more force, "
        "making them accelerate faster."
    )
    print(f"\n[2] Simulated student agrees and explains: '{simulated_explanation[:60]}…'")

    # 3. Diagnose
    print("\n[3] Running reasoning diagnostician…")
    diagnosis = diagnose_reasoning(concept, claim, agreed=True, explanation=simulated_explanation)
    print(f"    ✅ Student correct: {diagnosis['student_correct']}")
    print(f"       Misconception: {diagnosis['misconception']}")
    print(f"       Confidence: {diagnosis['misconception_confidence']:.0%}")
    print(f"       Diagnosis: {diagnosis['diagnosis'][:80]}…")

    # 4. Intervention
    print("\n[4] Generating intervention…")
    intervention = generate_intervention(concept, diagnosis["misconception"], diagnosis["diagnosis"])
    print(f"    ✅ Intervention: {intervention['intervention_question'][:80]}…")

    # 5. Recovery (simulating a correct follow-up answer)
    simulated_followup = (
        "In a vacuum both objects would fall at the same rate because gravity "
        "accelerates all objects equally regardless of mass — F=ma means the "
        "heavier object has more force but also more inertia, so they cancel out."
    )
    print(f"\n[5] Simulated follow-up answer: '{simulated_followup[:60]}…'")
    print("\n[5] Running recovery check…")
    recovery = check_recovery(
        concept,
        diagnosis["misconception"],
        intervention["intervention_question"],
        simulated_followup,
    )
    print(f"    ✅ Resolved: {recovery['resolved']}")
    print(f"       Status: {recovery['resolution_status']}")
    print(f"       Updated confidence: {recovery['updated_misconception_confidence']:.0%}")

    # 6. Learner update
    print("\n[6] Updating learner model…")
    test_learner = init_learner()
    mastery_before = test_learner["mastery"][concept]
    updated = update_learner(test_learner, concept, diagnosis, recovery["resolved"])
    mastery_after = updated["mastery"][concept]
    print(f"    ✅ Mastery: {mastery_before:.0%} → {mastery_after:.0%}")
    print(f"       Attempts recorded: {len(updated['attempts'])}")
    print(f"       Resolved: {updated['resolved'].get(concept, False)}")

    print(f"\n{'='*60}")
    print("✅ All components passed smoke test!")
    print(f"{'='*60}\n")


# Uncomment to run the smoke test:
# run_smoke_test()

---

## What TutorTrap Demonstrates

### 1. AI-Generated Realistic Misconceptions
The **Claim Generator** uses an LLM to produce plausible-but-incorrect physics claims grounded in real student misconception patterns — not random wrong answers, but believable statements that expose the kinds of reasoning errors students actually make.

### 2. AI Analysis of Free-Form Student Reasoning
The **Reasoning Diagnostician** doesn't just check if the student was right or wrong. It reads the student's explanation in natural language and identifies *what mental model* the student is using. This is qualitatively different from multiple-choice grading.

### 3. Specific Misconception Identification
Rather than saying "incorrect," TutorTrap names the specific misconception (*e.g., "Greater force automatically means greater acceleration"*), provides evidence from the student's own words, and assigns a confidence score. The student sees why they're wrong, not just that they're wrong.

### 4. Personalized Targeted Intervention
The **Intervention Generator** uses the diagnosed misconception to craft a follow-up question specifically designed to break that misconception. A student who believed heavier objects fall faster gets a question about vacuum conditions — not a generic question about gravity.

### 5. Measurable Learner State Change
The **Recovery Check** evaluates whether the follow-up answer shows real conceptual change. The **Adaptive Learner Model** updates mastery and misconception confidence, and shows the before/after delta. The full loop — *misconception detected → targeted intervention → misconception reduced* — is visible in a single session.

---

## Hackathon Demo Flow

**Target duration: ~2 minutes**

1. **Click "New Challenge"** — TutorTrap generates a validated misconception-based claim (e.g., *"A heavier object falls faster because gravity pulls harder on it."*)

2. **Click "Agree"** and type a plausible-but-wrong explanation (e.g., *"I agree because heavier objects have more mass, so gravity pulls them harder, making them accelerate faster."*) — then click **"Analyze My Reasoning."**

3. **TutorTrap shows the diagnosis:** named misconception, confidence score, evidence quoted from the student's own explanation, and a clear diagnosis — not just "wrong."

4. **A targeted follow-up question appears:** designed to specifically target the identified misconception (e.g., *"Two objects dropped in a vacuum — what happens?"*)

5. **Type a corrected answer** that demonstrates the right understanding. Click **"Submit Answer."**

6. **TutorTrap shows the recovery result:** resolved/partial/unresolved, before/after misconception confidence, before/after mastery.

7. **Open the Teacher View accordion** and click **"Refresh Chart"** to show the mastery bar chart with the concept now highlighted in green — learner state visibly changed.

---
*TutorTrap — SPEED September AI Challenge 2025*  
*Generative AI + Explainable Adaptive Learner Modeling*